In [123]:
# Cell 1: Imports and Setup
import mutalyzer
import pandas as pd
import numpy as np
import hgvs.parser
import hgvs.dataproviders.uta
import hgvs.assemblymapper
import sys
import gc
import os
import pickle
import time
import json
import warnings
from collections import defaultdict
from IPython.display import clear_output, display, HTML
import threading

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)



In [124]:
# Cell 2: Fixed HGVS Processor Class Definition
class HGVSProcessor:
    """HGVS Processor with resume capability for Jupyter Notebook"""
    
    def __init__(self, checkpoint_dir='./hgvs_checkpoints'):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)
        
        # File paths
        self.checkpoint_file = os.path.join(checkpoint_dir, 'processing_checkpoint.json')
        self.cache_file = os.path.join(checkpoint_dir, 'variant_cache.pkl')
        self.progress_file = os.path.join(checkpoint_dir, 'processing_progress.log')
        self.results_file = os.path.join(checkpoint_dir, 'partial_results.pkl')
        
        # State variables
        self.variant_cache = {}
        self.checkpoint = None
        self.is_interrupted = False
        
        # Initialize HGVS components
        self.hp, self.hdp, self.am = self._initialize_hgvs()
        
        # Load existing state
        self._load_state()
        
        print(f"✓ HGVS Processor initialized")
        print(f"  Checkpoint directory: {checkpoint_dir}")
        print(f"  Cache size: {len(self.variant_cache):,} variants")
        
    def _initialize_hgvs(self, max_retries=5):
        """Initialize HGVS components with retry logic"""
        for attempt in range(max_retries):
            try:
                print(f"Initializing HGVS components (attempt {attempt + 1}/{max_retries})...")
                hp = hgvs.parser.Parser()
                hdp = hgvs.dataproviders.uta.connect()
                am = hgvs.assemblymapper.AssemblyMapper(
                    hdp, 
                    assembly_name='GRCh38', 
                    alt_aln_method='splign'
                )
                print("✓ HGVS components initialized successfully")
                return hp, hdp, am
                
            except Exception as e:
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt  # Exponential backoff
                    print(f"  Connection failed: {e}")
                    print(f"  Retrying in {wait_time} seconds...")
                    time.sleep(wait_time)
                else:
                    print(f"✗ Failed to initialize HGVS after {max_retries} attempts")
                    raise
    
    def _load_state(self):
        """Load checkpoint, cache, and results"""
        # Load cache
        if os.path.exists(self.cache_file):
            try:
                with open(self.cache_file, 'rb') as f:
                    self.variant_cache = pickle.load(f)
                print(f"✓ Loaded {len(self.variant_cache):,} cached variants")
            except Exception as e:
                print(f"✗ Error loading cache: {e}")
                self.variant_cache = {}
        
        # Load checkpoint
        if os.path.exists(self.checkpoint_file):
            try:
                with open(self.checkpoint_file, 'r') as f:
                    self.checkpoint = json.load(f)
                print(f"✓ Found checkpoint from: {self.checkpoint.get('timestamp', 'unknown')}")
                print(f"  Phase: {self.checkpoint.get('phase', 1)}")
                print(f"  Processed unique variants: {len(self.checkpoint.get('processed_unique', [])):,}")
                print(f"  Processed rows: {len(self.checkpoint.get('processed_rows', [])):,}")
                print(f"  Progress: {self.checkpoint.get('progress_percent', 0):.1f}%")
            except Exception as e:
                print(f"✗ Error loading checkpoint: {e}")
                self.checkpoint = None
        
        # Load partial results if any
        if os.path.exists(self.results_file):
            try:
                with open(self.results_file, 'rb') as f:
                    self.partial_results = pickle.load(f)
                print(f"✓ Loaded partial results for {len(self.partial_results):,} rows")
            except:
                self.partial_results = {}
    
    def _save_checkpoint(self, phase, processed_unique_count, processed_rows_count, 
                        total_unique, total_rows, current_variant=None):
        """Save current processing state - FIXED VERSION"""
        try:
            # Calculate progress percentage safely
            if total_rows > 0:
                progress_percent = (processed_rows_count / total_rows) * 100
            else:
                progress_percent = 0
            
            checkpoint = {
                'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
                'phase': phase,
                'processed_unique_count': processed_unique_count,
                'processed_rows_count': processed_rows_count,
                'total_unique': total_unique,
                'total_rows': total_rows,
                'current_variant': current_variant,
                'progress_percent': progress_percent,
                'cache_size': len(self.variant_cache)
            }
            
            with open(self.checkpoint_file, 'w') as f:
                json.dump(checkpoint, f, indent=2)
            
            # Save cache periodically
            if processed_rows_count % 1000 == 0:
                self._save_cache()
                
        except Exception as e:
            print(f"✗ Error saving checkpoint: {e}")
    
    def _save_cache(self):
        """Save variant cache to disk"""
        try:
            with open(self.cache_file, 'wb') as f:
                pickle.dump(self.variant_cache, f)
        except Exception as e:
            print(f"✗ Error saving cache: {e}")
    
    def _save_results(self, dataframe, indices):
        """Save partial results to disk"""
        try:
            partial = {}
            for idx in indices:
                partial[idx] = dataframe.at[idx, 'hgvs_genome']
            
            with open(self.results_file, 'wb') as f:
                pickle.dump(partial, f)
        except Exception as e:
            print(f"✗ Error saving results: {e}")
    
    def _show_progress(self, processed, total, cache_hits, phase, 
                      current_variant=None, batch_info=None):
        """Display progress in Jupyter Notebook"""
        clear_output(wait=True)
        
        # Progress bar
        progress_percent = (processed / total * 100) if total > 0 else 0
        bar_length = 50
        filled_length = int(bar_length * processed // total)
        bar = '█' * filled_length + '░' * (bar_length - filled_length)
        
        # Create progress display
        html_content = f"""
        <div style="font-family: monospace; padding: 10px; border: 1px solid #ccc; border-radius: 5px; background: #f9f9f9;">
            <h3 style="margin-top: 0;">HGVS Processing - Phase {phase}</h3>
            
            <div style="margin: 10px 0;">
                <div style="display: flex; justify-content: space-between; margin-bottom: 5px;">
                    <span>Progress: {progress_percent:.1f}%</span>
                    <span>{processed:,} / {total:,}</span>
                </div>
                <div style="height: 20px; background: #ddd; border-radius: 10px; overflow: hidden;">
                    <div style="height: 100%; width: {progress_percent}%; background: #4CAF50; transition: width 0.3s;"></div>
                </div>
            </div>
            
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px; margin: 15px 0;">
                <div style="background: #e8f5e9; padding: 10px; border-radius: 5px;">
                    <strong>Cache Statistics</strong><br>
                    Hits: {cache_hits:,}<br>
                    Cache Size: {len(self.variant_cache):,}
                </div>
                
                <div style="background: #e3f2fd; padding: 10px; border-radius: 5px;">
                    <strong>Processing Info</strong><br>
                    Phase: {phase}<br>
                    {f'Batch: {batch_info}' if batch_info else ''}
                </div>
            </div>
            
            {f'<div style="background: #fff3e0; padding: 10px; border-radius: 5px; margin-top: 10px; font-size: 12px;"><strong>Current Variant:</strong><br>{current_variant[:80]}{"..." if len(current_variant) > 80 else ""}</div>' if current_variant else ''}
            
            <div style="margin-top: 15px; padding-top: 10px; border-top: 1px solid #ddd; font-size: 12px; color: #666;">
                Checkpoint saved automatically. Close and reopen notebook to resume.
            </div>
        </div>
        """
        
        display(HTML(html_content))
        
        # Also print to console for logging
        print(f"Phase {phase}: {processed:,}/{total:,} ({progress_percent:.1f}%) | "
              f"Cache: {cache_hits:,} hits | Cache size: {len(self.variant_cache):,}")
        if current_variant:
            print(f"Current: {current_variant[:100]}{'...' if len(current_variant) > 100 else ''}")
    
    def process_variant(self, variant_string):
        """Process a single variant string with error handling"""
        if not isinstance(variant_string, str) or pd.isna(variant_string):
            return None
        
        # Check cache first
        if variant_string in self.variant_cache:
            return self.variant_cache[variant_string]
        
        # Validate format
        if ':c.' not in variant_string:
            result = f"Error: Invalid format, expected ':c.' in {variant_string[:50]}..."
            self.variant_cache[variant_string] = result
            return result
        
        try:
            transcript_id = variant_string.split(':c.')[0]
            rest = variant_string.split(':c.')[1]
            
            # Parse multiple variants
            variants = []
            if '[' in rest and ']' in rest:
                inside_brackets = rest.split('[')[1].split(']')[0]
                variant_parts = [part.strip() for part in inside_brackets.split(';') if part.strip()]
                variants = [f"{transcript_id}:c.{part}" for part in variant_parts]
            elif ',' in rest and '[' not in rest:
                parts = [part.strip() for part in rest.split(',') if part.strip()]
                variants = [f"{transcript_id}:c.{part}" for part in parts]
            else:
                variants = [variant_string]
            
            genomic_components = []
            accession = None
            
            for var in variants:
                try:
                    parsed = self.hp.parse_hgvs_variant(var)
                    genomic = self.am.c_to_g(parsed)
                    genomic_str = str(genomic)
                    
                    if not accession:
                        accession = genomic_str.split(':g.')[0]
                    
                    position = genomic_str.split(':g.')[1]
                    genomic_components.append(position)
                    
                    del parsed
                    del genomic
                    
                except Exception as e:
                    result = f"Error in {var[:50]}: {str(e)[:100]}"
                    self.variant_cache[variant_string] = result
                    return result
            
            # Format output
            if not genomic_components:
                result = None
            elif len(genomic_components) == 1:
                result = f"{accession}:g.{genomic_components[0]}"
            else:
                result = f"{accession}:g.[{';'.join(genomic_components)}]"
            
            self.variant_cache[variant_string] = result
            return result
            
        except Exception as e:
            result = f"Error: {str(e)[:100]}"
            self.variant_cache[variant_string] = result
            return result
    
    def process_dataframe(self, dataframe, variant_column='hgvs_transcript', 
                         result_column='hgvs_genome', batch_size=1000,
                         resume=True, show_progress=True):
        """
        Process variants in a DataFrame with resume capability
        
        Parameters:
        -----------
        dataframe : pandas DataFrame
            DataFrame containing variant strings
        variant_column : str
            Column name containing HGVS transcript variants
        result_column : str
            Column name to store genomic results
        batch_size : int
            Number of variants to process before saving checkpoint
        resume : bool
            Whether to resume from checkpoint
        show_progress : bool
            Whether to display progress in notebook
            
        Returns:
        --------
        pandas DataFrame
            Updated DataFrame with genomic variants
        """
        
        print("="*70)
        print("HGVS TRANSCRIPT TO GENOMIC CONVERSION")
        print("="*70)
        
        # Validate input
        if variant_column not in dataframe.columns:
            raise ValueError(f"Column '{variant_column}' not found in DataFrame")
        
        # Initialize result column
        if result_column not in dataframe.columns:
            dataframe[result_column] = None
        
        # Filter variants of interest
        mask = dataframe[variant_column].astype(str).str.startswith('NM_000157.4')
        variant_series = dataframe.loc[mask, variant_column]
        total_rows = len(variant_series)
        
        if total_rows == 0:
            print("No variants starting with NM_000157.4 found")
            return dataframe
        
        # Get unique variants
        duplicate_counts = variant_series.value_counts()
        unique_variants_count = len(duplicate_counts)
        
        print(f"Total variants to process: {total_rows:,}")
        print(f"Unique variants: {unique_variants_count:,}")
        print(f"Duplication factor: {total_rows/unique_variants_count:.2f}x")
        print(f"Cache already contains: {len(self.variant_cache):,} variants")
        print("-"*70)
        
        # Determine resume point
        if resume and self.checkpoint:
            print("🔄 RESUMING FROM CHECKPOINT")
            phase = self.checkpoint.get('phase', 1)
            processed_unique_count = self.checkpoint.get('processed_unique_count', 0)
            processed_rows_count = self.checkpoint.get('processed_rows_count', 0)
            
            # We need to reconstruct which variants have been processed
            # This is simplified - in production you'd want to save the actual list
            processed_unique_set = set()
            
            # Filter remaining variants (simplified approach)
            unique_variants = duplicate_counts.index.tolist()
            
            # Since we don't know which exact ones were processed,
            # we'll start from the beginning but skip based on count
            start_index = min(processed_unique_count, len(unique_variants))
            unique_variants = unique_variants[start_index:]
            
            # For rows, we need to track which indices are done
            # We'll assume the first N rows were processed
            indices_to_process = variant_series.index.tolist()
            start_row_index = min(processed_rows_count, len(indices_to_process))
            indices_to_process = indices_to_process[start_row_index:]
            
            print(f"Already processed approximately: {processed_unique_count:,} unique variants")
            print(f"Already processed approximately: {processed_rows_count:,} rows")
            print(f"Remaining: {len(unique_variants):,} unique variants")
            print(f"Remaining: {len(indices_to_process):,} rows")
            
        else:
            print("🚀 STARTING NEW PROCESSING")
            phase = 1
            processed_unique_count = 0
            processed_rows_count = 0
            unique_variants = duplicate_counts.index.tolist()
            indices_to_process = variant_series.index.tolist()
            
            # Clear old checkpoint if starting fresh
            if os.path.exists(self.checkpoint_file):
                os.remove(self.checkpoint_file)
        
        # PHASE 1: Process unique variants
        if phase == 1 and unique_variants:
            print(f"\n📊 PHASE 1: Processing {len(unique_variants):,} unique variants")
            
            cache_hits = 0
            start_time = time.time()
            
            for i, variant_string in enumerate(unique_variants):
                # Process variant
                result = self.process_variant(variant_string)
                processed_unique_count += 1
                
                # Show progress
                if show_progress and (i % 10 == 0 or i == len(unique_variants) - 1):
                    self._show_progress(
                        processed_unique_count, 
                        unique_variants_count,
                        cache_hits,
                        phase=1,
                        current_variant=variant_string,
                        batch_info=f"{i+1}/{len(unique_variants)}"
                    )
                
                # Save checkpoint every batch_size
                if i % batch_size == 0 or i == len(unique_variants) - 1:
                    self._save_checkpoint(
                        phase=1,
                        processed_unique_count=processed_unique_count,
                        processed_rows_count=processed_rows_count,
                        total_unique=unique_variants_count,
                        total_rows=total_rows,
                        current_variant=variant_string
                    )
                    self._save_cache()
                    
                    if i > 0:
                        elapsed = time.time() - start_time
                        variants_per_second = (i + 1) / elapsed
                        remaining = len(unique_variants) - (i + 1)
                        eta = remaining / variants_per_second if variants_per_second > 0 else 0
                        
                        print(f"  Speed: {variants_per_second:.1f} variants/sec")
                        print(f"  ETA: {eta/60:.1f} minutes remaining")
            
            print(f"\n✅ PHASE 1 COMPLETE")
            print(f"  Processed {processed_unique_count:,} unique variants")
            print(f"  Time elapsed: {(time.time() - start_time)/60:.1f} minutes")
            
            # Move to phase 2
            phase = 2
            self._save_checkpoint(
                phase=2,
                processed_unique_count=processed_unique_count,
                processed_rows_count=processed_rows_count,
                total_unique=unique_variants_count,
                total_rows=total_rows,
                current_variant="Starting phase 2"
            )
        
        # PHASE 2: Apply cached results to all rows
        if phase == 2 and indices_to_process:
            print(f"\n📊 PHASE 2: Applying results to {len(indices_to_process):,} rows")
            
            cache_hits = 0
            start_time = time.time()
            results_indices = []
            
            for i, idx in enumerate(indices_to_process):
                variant_string = dataframe.at[idx, variant_column]
                
                # Get result from cache
                if variant_string in self.variant_cache:
                    result = self.variant_cache[variant_string]
                    cache_hits += 1
                else:
                    # Should not happen, but process if needed
                    result = self.process_variant(variant_string)
                
                # Update DataFrame
                dataframe.at[idx, result_column] = result
                processed_rows_count += 1
                results_indices.append(idx)
                
                # Show progress
                if show_progress and (i % 100 == 0 or i == len(indices_to_process) - 1):
                    self._show_progress(
                        processed_rows_count,
                        total_rows,
                        cache_hits,
                        phase=2,
                        current_variant=None,
                        batch_info=f"{i+1}/{len(indices_to_process)}"
                    )
                
                # Save checkpoint and results periodically
                if i % batch_size == 0 or i == len(indices_to_process) - 1:
                    self._save_checkpoint(
                        phase=2,
                        processed_unique_count=processed_unique_count,
                        processed_rows_count=processed_rows_count,
                        total_unique=unique_variants_count,
                        total_rows=total_rows,
                        current_variant=f"Row {i+1}/{len(indices_to_process)}"
                    )
                    
                    # Save partial results
                    self._save_results(dataframe, results_indices)
                    results_indices = []
                    
                    if i > 0:
                        elapsed = time.time() - start_time
                        rows_per_second = (i + 1) / elapsed
                        remaining = len(indices_to_process) - (i + 1)
                        eta = remaining / rows_per_second if rows_per_second > 0 else 0
                        
                        print(f"  Speed: {rows_per_second:.1f} rows/sec")
                        print(f"  ETA: {eta/60:.1f} minutes remaining")
                
                # Clean up memory
                if i % 5000 == 0:
                    gc.collect()
            
            print(f"\n✅ PHASE 2 COMPLETE")
            print(f"  Processed {processed_rows_count:,} rows")
            print(f"  Cache hits: {cache_hits:,} ({cache_hits/total_rows*100:.1f}%)")
            print(f"  Time elapsed: {(time.time() - start_time)/60:.1f} minutes")
            
            # Clean up checkpoint files
            if os.path.exists(self.checkpoint_file):
                os.remove(self.checkpoint_file)
                print(f"  Removed checkpoint file")
            
            if os.path.exists(self.results_file):
                os.remove(self.results_file)
                print(f"  Removed partial results file")
            
            # Save final cache
            self._save_cache()
        
        # Print summary statistics
        print("\n" + "="*70)
        print("📈 PROCESSING SUMMARY")
        print("="*70)
        
        # Count results
        results_series = dataframe.loc[mask, result_column]
        error_mask = results_series.astype(str).str.startswith('Error')
        valid_mask = results_series.notna() & ~error_mask
        
        error_count = error_mask.sum()
        valid_count = valid_mask.sum()
        null_count = results_series.isna().sum()
        
        print(f"Total variants processed: {total_rows:,}")
        print(f"Valid results: {valid_count:,} ({valid_count/total_rows*100:.1f}%)")
        print(f"Errors: {error_count:,} ({error_count/total_rows*100:.1f}%)")
        print(f"Null results: {null_count:,} ({null_count/total_rows*100:.1f}%)")
        print(f"Final cache size: {len(self.variant_cache):,} variants")
        
        # Show sample of errors if any
        if error_count > 0:
            print(f"\n❌ Sample errors:")
            error_samples = results_series[error_mask].head(5)
            for variant, error in zip(variant_series[error_mask].head(5), error_samples):
                print(f"  {variant[:60]}... → {error}")
        
        # Show sample of valid results
        if valid_count > 0:
            print(f"\n✅ Sample valid results:")
            valid_samples = results_series[valid_mask].head(5)
            for variant, result in zip(variant_series[valid_mask].head(5), valid_samples):
                print(f"  {variant[:60]}... → {result[:60]}...")
        
        print("\n💾 Cache saved for future runs")
        print("🔄 Run again to resume if interrupted")
        
        return dataframe
    
    def get_stats(self):
        """Get processing statistics"""
        stats = {
            'cache_size': len(self.variant_cache),
            'checkpoint_exists': os.path.exists(self.checkpoint_file),
            'checkpoint_dir': self.checkpoint_dir
        }
        
        if os.path.exists(self.checkpoint_file):
            try:
                with open(self.checkpoint_file, 'r') as f:
                    checkpoint = json.load(f)
                    stats.update({
                        'phase': checkpoint.get('phase'),
                        'processed_unique_count': checkpoint.get('processed_unique_count', 0),
                        'processed_rows_count': checkpoint.get('processed_rows_count', 0),
                        'timestamp': checkpoint.get('timestamp'),
                        'progress_percent': checkpoint.get('progress_percent', 0)
                    })
            except:
                pass
        
        return stats
    
    def clear_checkpoint(self):
        """Clear all checkpoint files (start fresh)"""
        files = [self.checkpoint_file, self.cache_file, 
                self.progress_file, self.results_file]
        
        for file in files:
            if os.path.exists(file):
                os.remove(file)
                print(f"Removed: {file}")
        
        self.checkpoint = None
        self.variant_cache = {}
        print("✅ All checkpoints cleared. Ready for fresh start.")

In [125]:
# Cell 3: Usage Example and Initialization
# Initialize processor
processor = HGVSProcessor(checkpoint_dir='./hgvs_checkpoints')

# Check current status
stats = processor.get_stats()
print("Current Status:")
for key, value in stats.items():
    print(f"  {key}: {value}")

Initializing HGVS components (attempt 1/5)...
✓ HGVS components initialized successfully
✓ Loaded 17,190 cached variants
✓ HGVS Processor initialized
  Checkpoint directory: ./hgvs_checkpoints
  Cache size: 17,190 variants
Current Status:
  cache_size: 17190
  checkpoint_exists: False
  checkpoint_dir: ./hgvs_checkpoints


In [126]:
# Cell 4: Process Your Variants
# Load your data
# Assuming you have a DataFrame called 'variants'
variants = pd.read_csv('processed_variants_with_genomic.csv', keep_default_na=False)
variants.head()

# Process with resume capability
# This will resume from where it left off if you close and reopen the notebook
if 'variants' in locals() and isinstance(variants, pd.DataFrame):
    print("Starting processing...")
    variants = processor.process_dataframe(
        variants,
        variant_column='hgvs_transcript',  # Change to your column name
        result_column='hgvs_genome',       # Column to store results
        batch_size=500,                     # Save checkpoint every 500 variants
        resume=True,                       # Resume from checkpoint
        show_progress=True                 # Show progress in notebook
    )
else:
    print("Please load your variants DataFrame first")


Phase 2: 123,668/123,668 (100.0%) | Cache: 123,668 hits | Cache size: 17,190
  Speed: 16868.7 rows/sec
  ETA: 0.0 minutes remaining

✅ PHASE 2 COMPLETE
  Processed 123,668 rows
  Cache hits: 123,668 (100.0%)
  Time elapsed: 0.1 minutes
  Removed checkpoint file
  Removed partial results file

📈 PROCESSING SUMMARY
Total variants processed: 123,668
Valid results: 123,668 (100.0%)
Errors: 0 (0.0%)
Null results: 0 (0.0%)
Final cache size: 17,190 variants

✅ Sample valid results:
  NM_000157.4:c.[233G>A;234C>G]... → NC_000001.11:g.[155239960C>T;155239959G>C]...
  NM_000157.4:c.[139A>T;141C>G]... → NC_000001.11:g.[155240054T>A;155240052G>C]...
  NM_000157.4:c.[523G>A;524C>G;525A>G]... → NC_000001.11:g.[155238582C>T;155238581G>C;155238580T>C]...
  NM_000157.4:c.[187G>T;188A>C;189C>T]... → NC_000001.11:g.[155240006C>A;155240005T>G;155240004G>A]...
  NM_000157.4:c.[55A>T;56G>T;57C>T]... → NC_000001.11:g.[155240690T>A;155240689C>A;155240688G>A]...

💾 Cache saved for future runs
🔄 Run again to re

In [127]:
# Cell 5: Updated Quick Check Functions
def check_status():
    """Check current processing status"""
    stats = processor.get_stats()
    
    if stats['checkpoint_exists']:
        print(f"📌 CHECKPOINT FOUND")
        print(f"   Phase: {stats.get('phase')}")
        print(f"   Progress: {stats.get('progress_percent', 0):.1f}%")
        print(f"   Processed unique variants: {stats.get('processed_unique_count', 0):,}")
        print(f"   Processed rows: {stats.get('processed_rows_count', 0):,}")
        print(f"   Last update: {stats.get('timestamp')}")
        print(f"\n   Cache contains: {stats['cache_size']:,} variants")
        
        resume_option = input("\nResume processing? (y/n): ")
        if resume_option.lower() == 'y':
            return 'resume'
        else:
            return 'stop'
    else:
        print("No checkpoint found. Ready for new processing.")
        return 'new'

In [128]:
# Cell 6: Resume After Interruption
# If you close your laptop or lose connection, just run this cell to resume

print("Checking for previous processing...")
status = check_status()

if status == 'resume':
    print("\nResuming processing...")
    # Re-initialize processor (will load checkpoint)
    processor = HGVSProcessor(checkpoint_dir='./hgvs_checkpoints')
    
    # Continue processing (make sure 'variants' DataFrame is loaded)
    if 'variants' in locals():
        variants = processor.process_dataframe(
            variants,
            variant_column='hgvs_transcript',
            result_column='hgvs_genome',
            resume=True,
            show_progress=True
        )
    else:
        print("Error: 'variants' DataFrame not found. Please load your data first.")
elif status == 'new':
    print("Starting new processing...")
    # Make sure your 'variants' DataFrame is loaded
    if 'variants' in locals():
        processor = HGVSProcessor(checkpoint_dir='./hgvs_checkpoints')
        variants = processor.process_dataframe(
            variants,
            variant_column='hgvs_transcript',
            result_column='hgvs_genome',
            resume=False,  # Start fresh
            show_progress=True
        )
    else:
        print("Please load your variants DataFrame first")
else:
    print("Processing stopped.")

Phase 2: 123,668/123,668 (100.0%) | Cache: 123,668 hits | Cache size: 17,190
  Speed: 16179.3 rows/sec
  ETA: 0.0 minutes remaining

✅ PHASE 2 COMPLETE
  Processed 123,668 rows
  Cache hits: 123,668 (100.0%)
  Time elapsed: 0.1 minutes
  Removed checkpoint file
  Removed partial results file

📈 PROCESSING SUMMARY
Total variants processed: 123,668
Valid results: 123,668 (100.0%)
Errors: 0 (0.0%)
Null results: 0 (0.0%)
Final cache size: 17,190 variants

✅ Sample valid results:
  NM_000157.4:c.[233G>A;234C>G]... → NC_000001.11:g.[155239960C>T;155239959G>C]...
  NM_000157.4:c.[139A>T;141C>G]... → NC_000001.11:g.[155240054T>A;155240052G>C]...
  NM_000157.4:c.[523G>A;524C>G;525A>G]... → NC_000001.11:g.[155238582C>T;155238581G>C;155238580T>C]...
  NM_000157.4:c.[187G>T;188A>C;189C>T]... → NC_000001.11:g.[155240006C>A;155240005T>G;155240004G>A]...
  NM_000157.4:c.[55A>T;56G>T;57C>T]... → NC_000001.11:g.[155240690T>A;155240689C>A;155240688G>A]...

💾 Cache saved for future runs
🔄 Run again to re

In [129]:
# Cell 8: Retry Network Errors Only
def retry_network_errors(processor, dataframe, variant_column='hgvs_transcript', result_column='hgvs_genome'):
    """
    Retry only variants with network-related errors, keeping other results intact
    """
    print("="*70)
    print("RETRYING NETWORK ERRORS ONLY")
    print("="*70)
    
    # Identify variants with network errors
    network_error_mask = (
        dataframe[result_column].astype(str).str.contains('Error', na=False)
    )
    
    # Also get variants with other errors (reference mismatch) - we'll keep these
    other_error_mask = (
        dataframe[result_column].astype(str).str.contains('Error:') &
        ~network_error_mask
    )
    
    network_error_count = network_error_mask.sum()
    other_error_count = other_error_mask.sum()
    valid_count = dataframe[result_column].notna().sum() - network_error_count - other_error_count
    
    print(f"Current status:")
    print(f"  Total variants: {len(dataframe):,}")
    print(f"  Valid results: {valid_count:,}")
    print(f"  Network errors (to retry): {network_error_count:,}")
    print(f"  Other errors (keep as is): {other_error_count:,}")
    
    if network_error_count == 0:
        print("\n✅ No network errors found. Nothing to retry.")
        return dataframe
    
    print(f"\n🔍 Identifying {network_error_count:,} variants with network errors...")
    
    # Get the variant strings with network errors
    network_error_variants = dataframe.loc[network_error_mask, variant_column].unique()
    print(f"  Found {len(network_error_variants):,} unique variants with network errors")
    
    # Remove these variants from cache so they'll be reprocessed
    print(f"\n🔄 Removing network-error variants from cache...")
    cache_removed = 0
    for variant in network_error_variants:
        if variant in processor.variant_cache:
            del processor.variant_cache[variant]
            cache_removed += 1
    
    print(f"  Removed {cache_removed:,} variants from cache")
    
    # Now process only these variants
    print(f"\n🚀 Retrying {len(network_error_variants):,} variants...")
    
    start_time = time.time()
    processed_count = 0
    cache_hits = 0  # Should be 0 since we removed them
    
    for i, variant_string in enumerate(network_error_variants):
        # Process variant (will fetch fresh from network)
        result = processor.process_variant(variant_string)
        processed_count += 1
        
        # Show progress
        if i % 10 == 0 or i == len(network_error_variants) - 1:
            clear_output(wait=True)
            progress_percent = (i + 1) / len(network_error_variants) * 100
            
            print("="*70)
            print(f"RETRYING NETWORK ERRORS: {progress_percent:.1f}%")
            print("="*70)
            print(f"Processed: {i+1:,}/{len(network_error_variants):,} variants")
            print(f"Current: {variant_string[:80]}{'...' if len(variant_string) > 80 else ''}")
            
            if i > 0:
                elapsed = time.time() - start_time
                variants_per_second = (i + 1) / elapsed
                remaining = len(network_error_variants) - (i + 1)
                eta = remaining / variants_per_second if variants_per_second > 0 else 0
                print(f"Speed: {variants_per_second:.1f} variants/sec")
                print(f"ETA: {eta/60:.1f} minutes remaining")
        
        # Save cache every 100 variants
        if i % 100 == 0 or i == len(network_error_variants) - 1:
            processor._save_cache()
    
    print(f"\n✅ Retry complete!")
    print(f"  Processed: {processed_count:,} variants")
    print(f"  Time: {(time.time() - start_time)/60:.1f} minutes")
    
    # Now update the dataframe with new results
    print(f"\n🔄 Updating DataFrame with new results...")
    
    updated_count = 0
    for idx in dataframe.index:
        variant_string = dataframe.at[idx, variant_column]
        result_column_value = dataframe.at[idx, result_column]
        
        # Check if this variant had a network error
        if isinstance(result_column_value, str) and (
            'bioutils.seqfetcher' in result_column_value or
            'network fetching' in result_column_value or
            'Failed to fetch' in result_column_value
        ):
            # Get new result from cache
            if variant_string in processor.variant_cache:
                new_result = processor.variant_cache[variant_string]
                dataframe.at[idx, result_column] = new_result
                updated_count += 1
    
    print(f"  Updated {updated_count:,} rows in DataFrame")
    
    # Final statistics
    print(f"\n📊 FINAL RESULTS:")
    
    new_network_error_mask = (
        dataframe[result_column].astype(str).str.contains('bioutils.seqfetcher', na=False) |
        dataframe[result_column].astype(str).str.contains('network fetching', na=False) |
        dataframe[result_column].astype(str).str.contains('Failed to fetch', na=False)
    )
    
    new_network_error_count = new_network_error_mask.sum()
    new_other_error_mask = (
        dataframe[result_column].astype(str).str.contains('Error:') &
        ~new_network_error_mask
    )
    new_other_error_count = new_other_error_mask.sum()
    new_valid_count = dataframe[result_column].notna().sum() - new_network_error_count - new_other_error_count
    
    print(f"  Valid results: {new_valid_count:,} ({new_valid_count/len(dataframe)*100:.1f}%)")
    print(f"  Network errors (remaining): {new_network_error_count:,} ({new_network_error_count/len(dataframe)*100:.1f}%)")
    print(f"  Other errors: {new_other_error_count:,} ({new_other_error_count/len(dataframe)*100:.1f}%)")
    
    if new_network_error_count > 0:
        print(f"\n⚠️  Still have {new_network_error_count:,} network errors.")
        print("You can run this function again if network connectivity improves.")
        print("Sample remaining network errors:")
        samples = dataframe.loc[new_network_error_mask, [variant_column, result_column]].head(3)
        for _, row in samples.iterrows():
            print(f"  {row[variant_column][:60]}... → {row[result_column][:60]}...")
    
    return dataframe




In [130]:
# Cell 9: Run Retry Function
if 'variants' in locals():
    print("Checking for network errors to retry...")
    
    # Make sure processor is initialized with your cache
    if 'processor' not in locals():
        processor = HGVSProcessor(checkpoint_dir='./hgvs_checkpoints')
    
    # Run the retry function
    variants = retry_network_errors(
        processor=processor,
        dataframe=variants,
        variant_column='hgvs_transcript',
        result_column='hgvs_genome'
    )
    
    # Save the updated results
    variants.to_csv('processed_variants_retry_network_errors.csv', index=False)
    print("\n💾 Results saved to 'processed_variants_retry_network_errors.csv'")
else:
    print("Please load your variants DataFrame first")

Checking for network errors to retry...
RETRYING NETWORK ERRORS ONLY
Current status:
  Total variants: 184,878
  Valid results: 184,878
  Network errors (to retry): 0
  Other errors (keep as is): 0

✅ No network errors found. Nothing to retry.

💾 Results saved to 'processed_variants_retry_network_errors.csv'


In [131]:
# Cell 10: Continuous Retry Function with Max Attempts
def continuous_retry_until_success(processor, dataframe, max_attempts=10, 
                                  variant_column='hgvs_transcript', 
                                  result_column='hgvs_genome',
                                  wait_between_attempts=30):
    """
    Keep retrying network errors until they're gone OR max attempts reached
    max_attempts: Maximum number of retry attempts (hard limit)
    wait_between_attempts: Seconds to wait between attempts (default 30)
    """
    
    print(f"\n{'='*70}")
    print(f"CONTINUOUS RETRY WITH MAX {max_attempts} ATTEMPTS")
    print(f"{'='*70}")
    
    # Track statistics
    stats = {
        'total_attempts': 0,
        'successful_retries': 0,
        'remaining_errors': 0,
        'attempts_made': []
    }
    
    for attempt in range(1, max_attempts + 1):
        print(f"\n📋 ATTEMPT {attempt}/{max_attempts}")
        print("-" * 50)
        
        # Count current network errors BEFORE this attempt
        network_error_mask = (
            dataframe[result_column].astype(str).str.contains('bioutils.seqfetcher', na=False) |
            dataframe[result_column].astype(str).str.contains('network fetching', na=False) |
            dataframe[result_column].astype(str).str.contains('Failed to fetch', na=False)
        )
        current_error_count = network_error_mask.sum()
        
        if current_error_count == 0:
            print(f"✅ SUCCESS! No more network errors.")
            print(f"   Total attempts made: {attempt-1}")
            if attempt > 1:
                print(f"   Errors fixed: {stats['successful_retries']:,}")
            break
            
        print(f"Network errors remaining: {current_error_count:,}")
        
        # Track starting point for this attempt
        starting_error_count = current_error_count
        
        # Wait between attempts (optional)
        if attempt > 1 and wait_between_attempts > 0:
            print(f"⏳ Waiting {wait_between_attempts} seconds before retry...")
            time.sleep(wait_between_attempts)
        
        # Retry network errors
        print(f"🔄 Retrying {current_error_count:,} variants...")
        dataframe = retry_network_errors(
            processor=processor,
            dataframe=dataframe,
            variant_column=variant_column,
            result_column=result_column
        )
        
        # Count errors AFTER retry
        network_error_mask_after = (
            dataframe[result_column].astype(str).str.contains('bioutils.seqfetcher', na=False) |
            dataframe[result_column].astype(str).str.contains('network fetching', na=False) |
            dataframe[result_column].astype(str).str.contains('Failed to fetch', na=False)
        )
        new_error_count = network_error_mask_after.sum()
        
        # Calculate progress
        errors_fixed = starting_error_count - new_error_count
        stats['successful_retries'] += errors_fixed
        stats['total_attempts'] = attempt
        stats['remaining_errors'] = new_error_count
        
        # Record attempt results
        stats['attempts_made'].append({
            'attempt': attempt,
            'errors_before': starting_error_count,
            'errors_after': new_error_count,
            'fixed': errors_fixed,
            'timestamp': time.strftime('%H:%M:%S')
        })
        
        print(f"\n📊 Attempt {attempt} Results:")
        print(f"   Errors before: {starting_error_count:,}")
        print(f"   Errors after: {new_error_count:,}")
        print(f"   Fixed this attempt: {errors_fixed:,}")
        
        if errors_fixed == 0 and attempt > 1:
            print(f"⚠️  No progress this attempt. Network may still be down.")
        
        # Save intermediate checkpoint
        if attempt < max_attempts or new_error_count == 0:
            temp_filename = f'processed_variants_after_attempt_{attempt}.csv'
            dataframe.to_csv(temp_filename, index=False)
            print(f"💾 Intermediate results saved to '{temp_filename}'")
        
        # Check if we've hit max attempts
        if attempt == max_attempts and new_error_count > 0:
            print(f"\n🚨 MAXIMUM ATTEMPTS REACHED ({max_attempts})")
            print(f"   Still have {new_error_count:,} network errors remaining")
            print(f"   Consider trying again later or accepting partial results")
            print(f"   Total fixed: {stats['successful_retries']:,}")
            
            # Show error breakdown
            unique_errors = dataframe.loc[network_error_mask_after, result_column].value_counts().head(5)
            print(f"\n📋 Top error types remaining:")
            for error_msg, count in unique_errors.items():
                short_msg = error_msg[:80] + "..." if len(error_msg) > 80 else error_msg
                print(f"   {short_msg}: {count:,}")
            break
    
    print(f"\n{'='*70}")
    print(f"RETRY PROCESS COMPLETE")
    print(f"{'='*70}")
    print(f"Total attempts made: {stats['total_attempts']}")
    print(f"Total errors fixed: {stats['successful_retries']:,}")
    print(f"Remaining network errors: {stats['remaining_errors']:,}")
    
    if stats['remaining_errors'] > 0:
        remaining_percent = (stats['remaining_errors'] / len(dataframe)) * 100
        print(f"Success rate: {(len(dataframe) - stats['remaining_errors']) / len(dataframe) * 100:.1f}%")
        print(f"⚠️  Note: {remaining_percent:.1f}% of variants still have network errors")
    
    return dataframe

In [132]:
# Cell 11: Run Continuous Retry with Max 10 Attempts
if 'variants' in locals():
    print("Starting continuous retry for network errors (max 10 attempts)...")
    
    # Check current status first
    network_error_mask = (

        variants['hgvs_genome'].astype(str).str.contains('Error', na=False)
    )
    
    current_errors = network_error_mask.sum()
    if current_errors == 0:
        print("✅ No network errors found. Nothing to retry.")
    else:
        print(f"Found {current_errors:,} network errors to retry.")
        
        # Ask for confirmation
        confirm = input(f"\nProceed with up to 10 retry attempts? (y/n): ")
        if confirm.lower() == 'y':
            if 'processor' not in locals():
                processor = HGVSProcessor(checkpoint_dir='./hgvs_checkpoints')
            
            # Run with max 10 attempts, 30 seconds between attempts
            variants = continuous_retry_until_success(
                processor=processor,
                dataframe=variants,
                max_attempts=10,          # Hard limit of 10 attempts
                variant_column='hgvs_transcript',
                result_column='hgvs_genome',
                wait_between_attempts=30   # Wait 30 seconds between attempts
            )
            
            # Ask if user wants to save
            save_now = input(f"\nSave results now? (y/n): ")
            if save_now.lower() == 'y':
                timestamp = time.strftime('%Y%m%d_%H%M%S')
                filename = f'parkin_variants_after_retries_{timestamp}.csv'
                variants.to_csv(filename, index=False)
                print(f"💾 Results saved to '{filename}'")
        else:
            print("Retry cancelled.")
else:
    print("Please load your variants DataFrame first")

Starting continuous retry for network errors (max 10 attempts)...
✅ No network errors found. Nothing to retry.
